<a href="https://colab.research.google.com/github/doomchitdoomchit/OD_DAY_SIDO_SEOULGU/blob/master/00_AB_test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 1. 가설 설계
- 그룹 정의
    - Group A (병목 집중기) : 주말 및 공휴일 직후 2영업일
    - Group B (평시 안정기) : 주말·공휴일 직후 2영업일을 제외한 평일
    - 제외 대상: 토요일, 일요일, 법정 공휴일

- 핵심 평가지표(KPI)
    - 평균 지표 : 자치구별 일평균 착지 물동량, 증가율
    - 변동성 지표 : 표준편차, 변동계수

- 가설 설정
    - 평균 검정($H_1$) : 자치구별 Group A의 일평균 물동량은 Group B보다 통계적으로 유의미하게 높다
    - 분산 검정 ($H_1$): Group A와 Group B의 물동량 분산에는 유의미한 차이가 있다

In [ ]:
import os
import holidays

import polars as pl
import pandas as pd
import numpy as np
import lightgbm as lgb

from scipy import stats
from datetime import timedelta
from sklearn.metrics import mean_squared_error, mean_absolute_error

from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# 데이터 로드
drive_path = '/content/drive/My Drive/Colab Notebooks/지역별 상품 물동량 통계 데이터'
data_name = 'seoulgu.parquet'

lazy_df = pl.scan_parquet(os.path.join(drive_path, data_name))

In [ ]:
# 준비 : col columns, 휴무일
all_cols = lazy_df.columns
cat_cols = [c for c in all_cols if c.startswith('CAT_VOL_')]

kr_holidays = list(holidays.KR(years=range(2019, 2023 + 1)).keys())

/tmp/ipykernel_611/3683647068.py:2: PerformanceWarning: Determining the column names of a LazyFrame requires resolving its schema, which is a potentially expensive operation. Use `LazyFrame.collect_schema().names()` to get the column names without this warning.
  all_cols = lazy_df.columns


In [ ]:
lazy_processed = lazy_df.with_columns([
    pl.sum_horizontal(cat_cols).alias('total_vol'),
    pl.col('DELIVERY_YMD').cast(pl.Date).alias('DELIVERY_YMD'),
]).group_by(['DELIVERY_YMD', 'RECEIVER_SIGUNGU_NM']).agg([
    pl.col('total_vol').sum(),
]).sort(['RECEIVER_SIGUNGU_NM', 'DELIVERY_YMD'])


lazy_processed = lazy_processed.with_columns([
    (pl.col('DELIVERY_YMD').dt.weekday() == 7).alias('is_sunday'),
        ((pl.col('DELIVERY_YMD').dt.weekday() == 7) |
         (pl.col('DELIVERY_YMD').dt.weekday() == 6) |
         (pl.col('DELIVERY_YMD').is_in(kr_holidays))
        ).cast(pl.Int64).alias('is_off_day')
]).with_columns([
    (pl.col('is_off_day') != pl.col('is_off_day').shift(1))
    .fill_null(True)
    .cum_sum()
    .alias('off_block_id'),
]).with_columns([
    pl.when(
        (pl.col('is_off_day') == 1) &
         (pl.col('is_sunday').any().over('off_block_id') | # 같은 연속 구간(off_block_id) 안에 일요일이 하루라도 있거나,
          (pl.len().over('off_block_id') >= 2)             # 같은 연속 구간의 전체 길이가 2일 이상인 경우
        )
    )
    .then(pl.len().over('off_block_id'))                     # [5] 참(True)일 때: 해당 연속 구간의 총 일수를 값으로 부여
    .otherwise(0)                                            # [6] 거짓(False)일 때: 영업일이거나 평일 하루짜리 공휴일이면 0 부여
    .alias('consecutive_off_days')
])

In [ ]:
lazy_processed = lazy_processed.with_columns([
    (pl.col('consecutive_off_days') > 0).alias('is_valid_holiday')
]).with_columns([
    # 2단계: 연휴 직후 1일차 / 2일차 / 나머지일 구분
    pl.when((pl.col('is_off_day') == 0) & pl.col('is_valid_holiday').shift(1).fill_null(False))
    .then(pl.lit('Group_A'))          # 연휴 직후 1일차 (가장 물량이 몰리는 날)

    .when((pl.col('is_off_day') == 0) & pl.col('is_valid_holiday').shift(2).fill_null(False))
    .then(pl.lit('Group_A'))          # 연휴 직후 2일차 (잔여 물량 해소일)

    .when(pl.col('is_off_day') == 0)
    .then(pl.lit('Group_B'))             # 나머지 일반 영업일

    .otherwise(pl.lit('off_day'))               # 휴무일
    .alias('day_type')
])

In [ ]:
lazy_processed = lazy_processed.filter(pl.col('day_type') != 'off_day')

In [ ]:
data = lazy_processed.select(['DELIVERY_YMD', 'RECEIVER_SIGUNGU_NM', 'total_vol', 'day_type']).collect()

In [ ]:
daily_data = data.group_by(['DELIVERY_YMD', 'RECEIVER_SIGUNGU_NM']).agg(
    pl.col('total_vol').sum(),
    pl.col('day_type').first()
).to_pandas()

In [ ]:
results = []

for _GU_NM, _GU_data in daily_data.groupby('RECEIVER_SIGUNGU_NM', observed=True):
    a_vol = _GU_data[_GU_data['day_type'] == 'Group_A']['total_vol']
    b_vol = _GU_data[_GU_data['day_type'] == 'Group_B']['total_vol']

    # 평균 및 표준편차
    mean_a, mean_b = a_vol.mean(), b_vol.mean()
    std_a, std_b = a_vol.std(), b_vol.std()
    lift_pct = ((mean_a - mean_b) / mean_b) * 100
     # 분산 차이 검정 (Levene) & 평균 차이 검정 (Welch's t-test)
    _, p_var = stats.levene(a_vol, b_vol)
    _, p_mean = stats.ttest_ind(a_vol, b_vol, equal_var=False)

    # 맨-휘트니 U 검정 (Mann-Whitney U Test)
    u_stat, p_mwu = stats.mannwhitneyu(a_vol, b_vol, alternative='two-sided')

    # 효과 크기 (Cohen's d)
    pooled_std = np.sqrt((std_a**2 + std_b**2) / 2)
    cohens_d = (mean_a - mean_b) / pooled_std
    # 비모수 효과 크기: Rank-Biserial Correlation (r)
    n_a, n_b = len(a_vol), len(b_vol)
    rank_biserial_r = (2 * u_stat) / (n_a * n_b) - 1

    results.append({
      '자치구': _GU_NM,
      'GroupA_일평균': round(mean_a, 1),
      'GroupB_일평균': round(mean_b, 1),
      '병목_증가율(%)': round(lift_pct, 1),
      '평균검정_p값': round(p_mean, 4),
      'GroupA_표준편차': round(std_a, 1),
      'GroupB_표준편차': round(std_b, 1),
      '분산검정_p값': round(p_var, 4),
      '효과크기(Cohens_d)': round(cohens_d, 2),
      'MWU_p값': round(p_mwu, 4),
      '비모수_효과크기(r)': round(rank_biserial_r, 3),
    })

res_df = pd.DataFrame(results).sort_values(by='병목_증가율(%)', ascending=False)
print(res_df.to_string(index=False))

 자치구  GroupA_일평균  GroupB_일평균  병목_증가율(%)  평균검정_p값  GroupA_표준편차  GroupB_표준편차  분산검정_p값  효과크기(Cohens_d)  MWU_p값  비모수_효과크기(r)
 관악구     71748.3     49408.4       45.2      0.0      18346.4       9378.0      0.0            1.53     0.0        0.759
 양천구     58292.2     40208.8       45.0      0.0      15463.3       7928.9      0.0            1.47     0.0        0.727
 동작구     56349.2     38850.5       45.0      0.0      14396.4       7412.3      0.0            1.53     0.0        0.755
 노원구     65096.9     45046.7       44.5      0.0      17048.2       8732.6      0.0            1.48     0.0        0.731
 은평구     59594.5     41341.4       44.2      0.0      15803.7       8062.5      0.0            1.45     0.0        0.722
서대문구     46765.1     32473.0       44.0      0.0      11967.1       6304.6      0.0            1.49     0.0        0.741
 강동구     60787.8     42337.8       43.6      0.0      16799.2       9052.3      0.0            1.37     0.0        0.699
 도봉구     38131.0     26633.7    

In [ ]:
def assign_period(year):
  if year == 2019:
    return '팬데믹이전(2019)'
  elif year == 2023:
    return '엔데믹(2023)'
  return '팬데믹(2020-2022)'

In [ ]:
daily_data['year_type'] = daily_data['DELIVERY_YMD'].dt.year.apply(assign_period)

In [ ]:
results = []

for (_PERIOD, _GU_NM), _GU_data in daily_data.groupby(['year_type', 'RECEIVER_SIGUNGU_NM'], observed=True):
    a_vol = _GU_data[_GU_data['day_type'] == 'Group_A']['total_vol']
    b_vol = _GU_data[_GU_data['day_type'] == 'Group_B']['total_vol']

    # 평균 및 표준편차
    mean_a, mean_b = a_vol.mean(), b_vol.mean()
    std_a, std_b = a_vol.std(), b_vol.std()
    lift_pct = ((mean_a - mean_b) / mean_b) * 100
     # 분산 차이 검정 (Levene) & 평균 차이 검정 (Welch's t-test)
    _, p_var = stats.levene(a_vol, b_vol)
    _, p_mean = stats.ttest_ind(a_vol, b_vol, equal_var=False)

    # 맨-휘트니 U 검정 (Mann-Whitney U Test)
    u_stat, p_mwu = stats.mannwhitneyu(a_vol, b_vol, alternative='two-sided')

    # 효과 크기 (Cohen's d)
    pooled_std = np.sqrt((std_a**2 + std_b**2) / 2)
    cohens_d = (mean_a - mean_b) / pooled_std
    # 비모수 효과 크기: Rank-Biserial Correlation (r)
    n_a, n_b = len(a_vol), len(b_vol)
    rank_biserial_r = (2 * u_stat) / (n_a * n_b) - 1

    results.append({
      '시기': _PERIOD,
      '자치구': _GU_NM,
      'GroupA_일평균': round(mean_a, 1),
      'GroupB_일평균': round(mean_b, 1),
      '병목_증가율(%)': round(lift_pct, 1),
      '평균검정_p값': round(p_mean, 4),
      'GroupA_표준편차': round(std_a, 1),
      'GroupB_표준편차': round(std_b, 1),
      '분산검정_p값': round(p_var, 4),
      '효과크기(Cohens_d)': round(cohens_d, 2),
      'MWU_p값': round(p_mwu, 4),
      '비모수_효과크기(r)': round(rank_biserial_r, 3),
    })

res_df = pd.DataFrame(results).sort_values(by='병목_증가율(%)', ascending=False)
print(res_df.to_string(index=False))

            시기  자치구  GroupA_일평균  GroupB_일평균  병목_증가율(%)  평균검정_p값  GroupA_표준편차  GroupB_표준편차  분산검정_p값  효과크기(Cohens_d)  MWU_p값  비모수_효과크기(r)
팬데믹(2020-2022)  관악구     76263.6     52257.5       45.9      0.0      18790.1       8766.6   0.0000            1.64     0.0        0.827
팬데믹(2020-2022)  양천구     62709.4     42999.6       45.8      0.0      15651.9       7206.5   0.0000            1.62     0.0        0.816
   팬데믹이전(2019)  동작구     48740.0     33444.9       45.7      0.0      11561.9       6089.9   0.0000            1.66     0.0        0.827
팬데믹(2020-2022)  동작구     59862.6     41083.2       45.7      0.0      14722.7       6874.0   0.0000            1.63     0.0        0.827
   팬데믹이전(2019)  관악구     62686.6     43020.1       45.7      0.0      14883.7       7901.1   0.0000            1.65     0.0        0.825
팬데믹(2020-2022)  노원구     69437.8     47850.8       45.1      0.0      17427.4       8084.3   0.0000            1.59     0.0        0.799
팬데믹(2020-2022)  은평구     63850.5     44032.2     

In [ ]:
# [출력 1-1] 자치구별 3개 시기 병목 증가율(%) 비교 피벗 테이블
pivot_lift = res_df.pivot(
    index='자치구', columns='시기', values='병목_증가율(%)'
)
pivot_lift['팬데믹_증감(2기-1기)'] = (
    pivot_lift['팬데믹(2020-2022)'] - pivot_lift['팬데믹이전(2019)']
).round(1)
pivot_lift['엔데믹_완화폭(3기-2기)'] = (
    pivot_lift['엔데믹(2023)'] - pivot_lift['팬데믹(2020-2022)']
).round(1)
pivot_lift['최종_변화폭(3기-1기)'] = (
    pivot_lift['엔데믹(2023)'] - pivot_lift['팬데믹이전(2019)']
).round(1)

print('=== [표 1] 자치구별 3개 시기 월·화요일 병목 증가율(%) 변화 ===')
print(pivot_lift.sort_values(by='엔데믹(2023)', ascending=False))

=== [표 1] 자치구별 3개 시기 월·화요일 병목 증가율(%) 변화 ===
시기    엔데믹(2023)  팬데믹(2020-2022)  팬데믹이전(2019)  팬데믹_증감(2기-1기)  엔데믹_완화폭(3기-2기)  \
자치구                                                                           
양천구        43.7            45.8         44.5            1.3            -2.1   
노원구        43.2            45.1         44.8            0.3            -1.9   
관악구        43.2            45.9         45.7            0.2            -2.7   
동작구        43.1            45.7         45.7            0.0            -2.6   
은평구        42.8            45.0         43.8            1.2            -2.2   
서대문구       42.4            44.7         44.6            0.1            -2.3   
강동구        42.0            44.6         43.6            1.0            -2.6   
도봉구        42.0            43.8         42.9            0.9            -1.8   
송파구        41.5            43.8         42.7            1.1            -2.3   
강북구        41.2            42.5         41.1            1.4            -1.3   
서초구     

In [ ]:
# [출력 1-2] 자치구별 3개 시기 그룹 평균차 비교 피벗 테이블

res_df['그룹_평균차'] = res_df['GroupA_일평균'] - res_df['GroupB_일평균']

pivot_lift = res_df.pivot(
    index='자치구', columns='시기', values='그룹_평균차'
)
pivot_lift['팬데믹_증감(2기-1기)'] = (
    pivot_lift['팬데믹(2020-2022)'] - pivot_lift['팬데믹이전(2019)']
).round(1)
pivot_lift['엔데믹_완화폭(3기-2기)'] = (
    pivot_lift['엔데믹(2023)'] - pivot_lift['팬데믹(2020-2022)']
).round(1)
pivot_lift['최종_변화폭(3기-1기)'] = (
    pivot_lift['엔데믹(2023)'] - pivot_lift['팬데믹이전(2019)']
).round(1)

print('=== [표 1] 자치구별 3개 시기 월·화요일 평균차 변화 ===')
print(pivot_lift.sort_values(by='엔데믹(2023)', ascending=False))

=== [표 1] 자치구별 3개 시기 월·화요일 평균차 변화 ===
시기    엔데믹(2023)  팬데믹(2020-2022)  팬데믹이전(2019)  팬데믹_증감(2기-1기)  엔데믹_완화폭(3기-2기)  \
자치구                                                                           
강남구     29833.9         35010.0      28065.0         6945.0         -5176.1   
송파구     29128.5         33694.3      25943.9         7750.4         -4565.8   
강서구     23585.1         27775.5      21621.7         6153.8         -4190.4   
서초구     20859.4         24580.1      19158.7         5421.4         -3720.7   
관악구     20344.9         24006.1      19666.5         4339.6         -3661.2   
노원구     18513.7         21587.0      17315.2         4271.8         -3073.3   
마포구     18056.1         21210.9      16477.4         4733.5         -3154.8   
영등포구    17783.6         20389.2      15464.4         4924.8         -2605.6   
강동구     17666.6         20385.7      14022.6         6363.1         -2719.1   
은평구     17088.2         19818.3      15105.7         4712.6         -2730.1   
성북구     16989.

In [ ]:
import statsmodels.api as sm
from statsmodels.formula.api import ols
# [출력 2] 서울 전체 기준 시기(Period) x 요일그룹(Group) 이원 분산분석
seoul_daily = (
    daily_data.groupby(['DELIVERY_YMD', 'year_type', 'day_type'])['total_vol'].sum().reset_index()
)
model = ols('total_vol ~ C(year_type) + C(day_type) + C(year_type):C(day_type)', data=seoul_daily).fit()
print('\n=== [표 2] 서울 전체 Two-way ANOVA (시기(year_type) x 요일(day_type)그룹 상호작용 검정) ===')
print(sm.stats.anova_lm(model, typ=2))


=== [표 2] 서울 전체 Two-way ANOVA (시기(year_type) x 요일(day_type)그룹 상호작용 검정) ===
                                sum_sq      df           F         PR(>F)
C(year_type)              1.449511e+13     2.0  108.162622   5.119842e-44
C(day_type)               5.719330e+13     1.0  853.553861  3.969022e-143
C(year_type):C(day_type)  4.889848e+11     2.0    3.648809   2.630259e-02
Residual                  8.288653e+13  1237.0         NaN            NaN
